In [ ]:
import boto3
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Any
from datetime import datetime
import base64
from PIL import Image
import io

class BrainCTClarifyTester:
    def __init__(self, endpoint_name: str):
        self.endpoint_name = endpoint_name
        self.runtime_client = boto3.client('runtime.sagemaker', region_name='ap-southeast-1')
        
    def prepare_image_payload(self, image_url: str) -> Dict[str, Any]:
        """Prepare the payload for the endpoint"""
        return {
            "url": image_url,
            "explain": True  # Request SHAP explanations
        }
    
    def invoke_endpoint(self, image_url: str) -> Dict[str, Any]:
        """Invoke the endpoint and get predictions with explanations"""
        try:
            # Prepare payload
            payload = self.prepare_image_payload(image_url)
            
            # Invoke endpoint
            response = self.runtime_client.invoke_endpoint(
                EndpointName=self.endpoint_name,
                ContentType='application/json',
                Body=json.dumps(payload),
                CustomAttributes='accept_epl=true'  # Enable explanations
            )
            
            # Parse response
            result = json.loads(response['Body'].read().decode())
            return result
            
        except Exception as e:
            print(f"Error invoking endpoint: {str(e)}")
            raise
    
    def visualize_results(self, result: Dict[str, Any]):
        """Visualize the prediction and SHAP explanation"""
        try:
            # Extract prediction and explanation
            prediction = result.get('prediction')
            explanation = result.get('explanation')
            
            # Print prediction results
            print(f"Prediction Results:")
            print(f"Prediction: {prediction}")
            
            # If there's a SHAP explanation, visualize it
            if explanation:
                shap_values = np.array(explanation['shap_values'])
                
                # Create visualization
                plt.figure(figsize=(12, 4))
                
                # Plot original image
                plt.subplot(1, 2, 1)
                plt.title('Original Image')
                plt.imshow(result.get('input_image', np.zeros((32, 32))), cmap='gray')
                plt.axis('off')
                
                # Plot SHAP values
                plt.subplot(1, 2, 2)
                plt.title('SHAP Explanation')
                plt.imshow(shap_values.reshape(32, 32), cmap='RdBu')
                plt.colorbar(label='SHAP value')
                plt.axis('off')
                
                plt.tight_layout()
                plt.show()
                
        except Exception as e:
            print(f"Error visualizing results: {str(e)}")
            raise

# Test the endpoint
def test_clarify_endpoint(endpoint_name: str, test_image_url: str):
    """
    Test the Clarify endpoint with a single image
    
    Args:
        endpoint_name (str): Name of the deployed endpoint
        test_image_url (str): URL of the test image
    """
    try:
        # Initialize tester
        tester = BrainCTClarifyTester(endpoint_name)
        
        # Start time
        start_time = datetime.utcnow()
        print(f"Starting inference at: {start_time.strftime('%Y-%m-%d %H:%M:%S UTC')}")
        
        # Get prediction with explanation
        result = tester.invoke_endpoint(test_image_url)
        
        # End time
        end_time = datetime.utcnow()
        print(f"Finished inference at: {end_time.strftime('%Y-%m-%d %H:%M:%S UTC')}")
        print(f"Total time: {(end_time - start_time).total_seconds():.2f} seconds")
        
        # Visualize results
        tester.visualize_results(result)
        
        return result
        
    except Exception as e:
        print(f"Error testing endpoint: {str(e)}")
        raise

# Example usage:
test_image_url = "https://prod-images-static.radiopaedia.org/images/61513769/9131de5120872bcc1ad19d8198c759904a4b78efa405550c86ec0f19d3dda590_big_gallery.jpeg"  # Replace with your test image URL
endpoint_name = "brain-ct-clarify-20250429214831"  # Replace with your actual endpoint name

# Run the test
result = test_clarify_endpoint(endpoint_name, test_image_url)